# Пакет 3: трюки обучения — Даня ЖУКОВ (Kaggle T4, ~4.5ч)

Проверяем твои же приёмы в чистом виде (по одному, на одинаковом бюджете), чтобы
решить, что входит в рецепт финального большого прогона:

| Слот | Что проверяет |
|---|---|
| tiny_anchor | якорь без трюков |
| tiny_confw | confidence weighting: вес примера 0.75+0.25*|2y-1| (метки ~0.5 не выбрасываем, а ослабляем) |
| tiny_catbal | category-balanced loss: вес категории (median/count)^0.5, клип [0.65, 2.0] |
| tiny_confw_catbal | оба вместе (проверка сложения эффектов) |

Loss нормируется на сумму весов батча (не .mean()) — чтобы LR не плыл (правка Мишани).

## Запуск на Kaggle
1. Create Notebook -> File -> Import Notebook -> GitHub -> `zxcghole228/ecup-2026-product-matching`
   (ветка egor/llm-solution), файл notebooks/ablation_tricks_zhukov.ipynb.
2. Add Input -> датасет данных + файл data_polygon/llm_sample_2m.parquet.
3. Поправь BASE. Settings: GPU T4 x2, Internet On. Save & Run All.

Результат после каждого слота в json; читать: дельта full_macro к tiny_anchor,
порог включения в рецепт >= +0.003. Вернуть: json + строка в README + ноутбук в runs/.


In [ ]:
import os, json, gc, time, random, re
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import average_precision_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

BASE = "/kaggle/input/datasets/mihailivanovvvv/hakaton-ozon-math-items"   # <-- поправь
ITEMS_PATH = f"{BASE}/items.parquet"
MATCHES_LLM_PATH = f"{BASE}/matches_llm.parquet"
POLYGON_PATH = "/kaggle/input/datasets/w4stdd/2m-parquet/llm_sample_2m.parquet"  # датасет Егора, добавь в Input
RESULTS_PATH = "/kaggle/working/ablation_results.json"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda"
print(torch.cuda.get_device_name(0))

EXPERIMENTS = [
    dict(name="tiny_anchor",       text="v1", max_len=160, swap=False, confw=False, catbal=False, batch=256, lr=2e-4),
    dict(name="tiny_confw",        text="v1", max_len=160, swap=False, confw=True,  catbal=False, batch=256, lr=2e-4),
    dict(name="tiny_catbal",       text="v1", max_len=160, swap=False, confw=False, catbal=True,  batch=256, lr=2e-4),
    dict(name="tiny_confw_catbal", text="v1", max_len=160, swap=False, confw=True,  catbal=True,  batch=256, lr=2e-4),
]
MODEL_NAME = "cointegrated/rubert-tiny2"

## Данные, сплиты, атрибуты


In [ ]:
ml = pd.read_parquet(MATCHES_LLM_PATH)
parent = {}
def find(x):
    p = parent.setdefault(x, x)
    while p != parent[p]:
        parent[p] = parent[parent[p]]; p = parent[p]
    parent[x] = p; return p
for a, b in zip(ml.id1.values, ml.id2.values):
    ra, rb = find(a), find(b)
    if ra != rb: parent[rb] = ra
comp = np.fromiter((find(i) for i in ml.id1.values), dtype=np.int64, count=len(ml))
rng = np.random.RandomState(13)
uniq = np.unique(comp)
val_set = set(uniq[rng.rand(len(uniq)) < 0.03].tolist())
is_val = np.fromiter((c in val_set for c in comp), dtype=bool, count=len(ml))
holdout = ml[is_val].copy()
holdout = holdout[(holdout.target <= 0.2) | (holdout.target >= 0.8)]
holdout["target"] = (holdout.target >= 0.5).astype(np.int8)
del ml, parent, comp; gc.collect()

polygon = pd.read_parquet(POLYGON_PATH)
print(f"полигон {len(polygon):,}; holdout {len(holdout):,}")

need = set(polygon.id1) | set(polygon.id2) | set(holdout.id1) | set(holdout.id2)
raw = {}     # id -> (name, attrs_dict_normalized_keys, category)
def parse_attrs(a):
    try:
        d = json.loads(a) if isinstance(a, str) else {}
        return {str(k).lower(): str(v) for k, v in d.items() if v} if isinstance(d, dict) else {}
    except Exception:
        return {}
f = pq.ParquetFile(ITEMS_PATH)
for b in f.iter_batches(columns=["id", "name", "attributes", "category"], batch_size=500_000):
    df = b.to_pandas()
    df = df[df["id"].isin(need)]
    for i, n, a, c in df.itertuples(index=False, name=None):
        raw[i] = (str(n) if n is not None else "", parse_attrs(a), str(c) if c is not None else "")
holdout["category"] = [raw[i][2] for i in holdout.id1]
holdout_fast = holdout.sample(min(60_000, len(holdout)), random_state=0)
print(f"товаров: {len(raw):,}")

## Три варианта текста

- v1 — боевой (как в наших CE): name | приоритетные атрибуты, 260 симв, без нормализации.
- v2 — по ресерчу: категория в тексте, нормализация (lower, ё->е, х/×->x, ,->.), 35 ключей с фэшн (рост/обхват/пол/сезон), 460 симв.
- v3 — пересечение: общие ключи атрибутов ОБОИХ товаров идут первыми (парный, строится на лету).


In [ ]:
KEY_V1 = ["бренд", "артикул", "партномер", "oem", "код", "модель", "размер",
          "цвет", "объем", "обьем", "вес", "тип", "материал", "количество"]
KEY_V2 = ["бренд", "brand", "артикул", "партномер", "part number", "partnumber", "oem",
          "код", "sku", "модель", "model", "размер", "size", "рост", "обхват", "пол",
          "gender", "цвет", "color", "материал", "material", "сезон", "объем", "обьем",
          "volume", "вес", "weight", "длина", "ширина", "высота", "количество",
          "комплектация", "упаков", "тип", "type"]
MULT_RE = re.compile(r"[×хХ]")
SP_RE = re.compile(r"\s+")

def norm_piece(s):
    s = str(s).lower().replace("ё", "е")
    s = MULT_RE.sub("x", s).replace(",", ".")
    return SP_RE.sub(" ", s).strip()

def pick(attrs, key_order):
    picked, used = [], set()
    for want in key_order:
        for k, v in attrs.items():
            if want in k and k not in used:
                picked.append((k, v)); used.add(k)
    rest = [(k, v) for k, v in attrs.items() if k not in used]
    return picked + rest

def text_v1(item):
    name, attrs, cat = item
    s = " ; ".join(f"{k}:{v}" for k, v in pick(attrs, KEY_V1))[:260]
    return f"{name} | {s}"

def text_v2(item):
    name, attrs, cat = item
    s = " ; ".join(f"{k}: {norm_piece(v)}" for k, v in pick(attrs, KEY_V2))[:460]
    return f"категория: {norm_piece(cat)} | название: {norm_piece(name)} | атрибуты: {s}"

def pair_texts_v3(ia, ib):
    na, aa, ca = ia; nb, ab, cb = ib
    common = set(aa) & set(ab)
    def one(name, attrs):
        first = [(k, attrs[k]) for k, _ in pick({k: attrs[k] for k in common}, KEY_V2)]
        rest = [(k, v) for k, v in pick(attrs, KEY_V2) if k not in common]
        s = " ; ".join(f"{k}: {norm_piece(v)}" for k, v in first + rest)[:460]
        return f"название: {norm_piece(name)} | атрибуты: {s}"
    return one(na, aa), one(nb, ab)

TEXT_CACHE = {}
def get_text_fn(variant):
    if variant == "v3":
        return None  # парный, строится в Dataset
    if variant not in TEXT_CACHE:
        t0 = time.time()
        fn = text_v1 if variant == "v1" else text_v2
        TEXT_CACHE[variant] = {i: fn(it) for i, it in raw.items()}
        print(f"тексты {variant}: {time.time()-t0:.0f}s")
    return TEXT_CACHE[variant]

## Dataset / обучение / оценка


In [ ]:
class DS(Dataset):
    def __init__(self, df, variant, swap=False, training=False):
        self.a = df.id1.values; self.b = df.id2.values
        self.y = df.target.values.astype(np.float32)
        self.variant = variant; self.swap = swap and training
        self.texts = get_text_fn(variant)
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        a, b = self.a[i], self.b[i]
        if self.swap and random.random() < 0.5:
            a, b = b, a
        if self.variant == "v3":
            ta, tb = pair_texts_v3(raw[a], raw[b])
        else:
            ta, tb = self.texts[a], self.texts[b]
        cw = self.catw[i] if hasattr(self, "catw") else 1.0
        return ta, tb, self.y[i], cw

def macro_ap_df(df, preds):
    z = df[["category", "target"]].copy(); z["p"] = preds
    return float(z.groupby("category").apply(
        lambda g: average_precision_score(g.target, g.p)).mean())

@torch.no_grad()
def predict(model, tok, df, variant, max_len, bs=512):
    model.eval()
    dl = DataLoader(DS(df, variant), batch_size=bs, num_workers=0, shuffle=False,
                    collate_fn=lambda batch: tok([x[0] for x in batch], [x[1] for x in batch],
                        padding=True, truncation=True, max_length=max_len, return_tensors="pt"))
    out = []
    for enc in dl:
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.autocast("cuda", torch.float16):
            out.append(torch.sigmoid(model(**enc).logits.squeeze(-1).float()).cpu().numpy())
    return np.concatenate(out)

def run_experiment(cfg):
    t_start = time.time()
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1).to(device)
    # category-balanced веса (по train-части полигона)
    from collections import Counter
    if cfg.get("catbal"):
        cnt = Counter(raw[i][2] for i in polygon.id1.values)
        med = float(np.median(list(cnt.values())))
        catw = {c: float(np.clip((med / n) ** 0.5, 0.65, 2.0)) for c, n in cnt.items()}
        wsum = sum(cnt[c] * catw[c] for c in cnt)
        norm = wsum / sum(cnt.values())
        catw = {c: w / norm for c, w in catw.items()}
    def collate(batch):
        enc = tok([x[0] for x in batch], [x[1] for x in batch], padding=True,
                  truncation=True, max_length=cfg["max_len"], return_tensors="pt")
        y = torch.tensor([x[2] for x in batch])
        w = torch.ones_like(y)
        if cfg.get("confw"):
            w = w * (0.75 + 0.25 * (2 * y - 1).abs())
        if cfg.get("catbal"):
            w = w * torch.tensor([x[3] for x in batch])
        return enc, y, w
    ds = DS(polygon, cfg["text"], swap=cfg["swap"], training=True)
    if cfg.get("catbal"):
        ds.catw = [catw.get(raw[i][2], 1.0) for i in polygon.id1.values]
    dl = DataLoader(ds, batch_size=cfg["batch"], shuffle=True, num_workers=0,
                    drop_last=True, collate_fn=collate)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, len(dl)//30, len(dl))
    scaler = torch.amp.GradScaler()
    model.train(); t0 = time.time(); seen = 0
    for enc, y, w in dl:
        enc = {k: v.to(device, non_blocking=True) for k, v in enc.items()}
        y = y.to(device, non_blocking=True); w = w.to(device, non_blocking=True)
        with torch.autocast("cuda", torch.float16):
            per_ex = nn.functional.binary_cross_entropy_with_logits(
                model(**enc).logits.squeeze(-1), y, reduction="none")
            loss = (per_ex * w).sum() / w.sum().clamp_min(1e-6)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update(); sched.step()
        seen += len(y)
        if seen % (cfg["batch"] * 800) < cfg["batch"]:
            print(f"  [{cfg['name']}] {seen:,}/{len(polygon):,} {seen/(time.time()-t0):.0f} pair/s", flush=True)
    fast = macro_ap_df(holdout_fast, predict(model, tok, holdout_fast, cfg["text"], cfg["max_len"]))
    full = macro_ap_df(holdout, predict(model, tok, holdout, cfg["text"], cfg["max_len"]))
    res = dict(cfg, fast_macro=round(fast, 4), full_macro=round(full, 4),
               minutes=round((time.time()-t_start)/60, 1))
    del model; gc.collect(); torch.cuda.empty_cache()
    return res

In [ ]:
results = []
if os.path.exists(RESULTS_PATH):
    results = json.load(open(RESULTS_PATH))
    print("уже готово:", [r["name"] for r in results])
done = {r["name"] for r in results}
for cfg in EXPERIMENTS:
    if cfg["name"] in done:
        continue
    print(f"=== {cfg['name']} ===", flush=True)
    try:
        res = run_experiment(cfg)
    except Exception as e:
        print(f"  !! {cfg['name']}: {type(e).__name__}: {e} — пропускаю", flush=True)
        gc.collect(); torch.cuda.empty_cache()
        continue
    results.append(res)
    json.dump(results, open(RESULTS_PATH, "w"), ensure_ascii=False, indent=1)
    print(pd.DataFrame(results)[["name","confw","catbal","fast_macro","full_macro","minutes"]]
          .to_string(index=False), flush=True)
print("ГОТОВО")

## Как читать

- Сравнивать full_macro попарно с якорем tiny_v1_160.
- v2/v3 лучше якоря на >=0.004 -> формат едет в финальный прогон.
- 224 лучше на >=0.004 -> берём длину (цена: скорость инференса).
- swap: если не хуже якоря — включаем в финальный рецепт (даёт устойчивость к порядку + встроенный TTA).
- Результаты: строка в README + исполненный ноутбук в notebooks/runs/.
